In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [47]:
df = pd.read_csv("./data/Nepali_house_dataset.csv")

In [48]:
df_clean = df.copy()

In [49]:
df['PRICE'] = df['PRICE'].astype(str)

# Starting Over

In [72]:
df['PRICE_CLEAN'] = df['PRICE'].astype(str).str.strip()

In [75]:
# Identify non-price rows first
non_price_mask = df['PRICE_CLEAN'].str.contains('call|contact|negotiable', na=False, case=False)
print(f"non price rows: {non_price_mask.sum()}")
print(df[non_price_mask]['PRICE'].unique())

non price rows: 190
['Price on call' 'Price on call ']


In [78]:
# Identify rate prices (which contains '/')
rate_mask = df['PRICE_CLEAN'].str.contains('/', na=False, regex=False)
print(f"Rates row: {rate_mask.sum()}")
print(df[rate_mask]['PRICE'].unique())
print(df[rate_mask]['PRICE'].head())


Rates row: 561
['Rs. 50 Lac/aana' 'Rs. 65,000 /m' 'Rs. 1.4 Lac/m' 'Rs. 1.5 Lac/m'
 'Rs. 1.05 Lac/m' 'Rs. 90 Lac/aana' 'Rs. 2.5 Lac/m' 'Rs. 70 Lac/aana'
 'Rs. 21 Lac/aana' 'Rs. 60,000 /m' 'Rs. 3.5 Lac/m' 'Rs. 80,000 /m'
 'Rs. 70,000 /m' 'Rs. 45,000 /m' 'Rs. 1.6 Lac/m' 'Rs. 1.28 Lac/m'
 'Rs. 90,000 /m' 'Rs. 1.7 Lac/m' 'Rs. 1.85 Lac/m' 'Rs. 3 Lac/m'
 'Rs. 1.55 Lac/m' 'Rs. 2 Lac/m' 'Rs. 82,000 /m' 'Rs. 1.3 Lac/m'
 'Rs. 1 Lac/m' 'Rs. 2.2 Lac/m' 'Rs. 5 Lac/m' 'Rs. 1.75 Lac/m'
 'Rs. 75,000 /m' 'Rs. 1.1 Lac/m' 'Rs. 4 Lac/m' 'Rs. 6 Lac/m'
 'Rs. 1.65 Lac/m' 'Rs. 2.25 Lac/m' 'Rs. 3.35 Lac/m' 'Rs. 4.5 Lac/m'
 'Rs. 1.25 Lac/m' 'Rs. 12.7 Lac/m' 'Rs. 2.75 Lac/m' 'Rs. 85,000 /m'
 'Rs. 8 Lac/m' 'Rs. 2.9 Lac/m' 'Rs. 50,000 /m' 'Rs. 78,900 /m'
 'Rs. 15 Lac/m' 'Rs. 2.64 Lac/m' 'Rs. 1.35 Lac/m' 'Rs. 3.25 Lac/m'
 'Rs. 1.2 Lac/m' 'Rs. 1.8 Lac/m' 'Rs. 55,000 /m' 'Rs. 2.3 Lac/m'
 'Rs. 42,000 /m' 'Rs. 3.1 Lac/m' 'Rs. 8.5 Lac/m' 'Rs. 7.25 Lac/m'
 'Rs. 3.85 Lac/m' 'Rs. 2.1 Lac/m' 'Rs. 53 Lac/aana' 'Rs. 1.15 Lac/m

In [81]:
# Identify absolute Cr prices (which donesnot contain '/', but has 'Cr')
cr_absolute_mask = (~rate_mask) & (~non_price_mask) & df['PRICE_CLEAN'].str.contains('Cr', na=False, case=False, regex=False)
print(f"Cr absolute rows: {cr_absolute_mask.sum()}")
print(df[cr_absolute_mask]['PRICE'].head())

Cr absolute rows: 2431
0     Rs. 2.9 Cr 
1    Rs. 4.75 Cr 
2    Rs. 1.99 Cr 
3       Rs. 4 Cr 
6     Rs. 3.3 Cr 
Name: PRICE, dtype: object


In [87]:
# Identify absolute Lac prices (no '/', has 'Lac', no 'Cr')
lac_absolute_mask = (~rate_mask) & (~non_price_mask) & (~cr_absolute_mask) & df['PRICE_CLEAN'].str.contains('Lac', na=False, case=False, regex=False)

print(f"Lac absolute rows: {lac_absolute_mask.sum()}")
print(df[lac_absolute_mask]['PRICE'].head())

Lac absolute rows: 28
210    Rs. 1.5 Lac 
357     Rs. 90 Lac 
386     Rs. 65 Lac 
546     Rs. 15 Lac 
642    Rs. 1.1 Lac 
Name: PRICE, dtype: object


In [91]:
# Identify raw NPR absolute prices (no '/', no 'Cr', no 'Lac', has numbers only)
raw_npr_mask = (~rate_mask) & (~non_price_mask) & (~cr_absolute_mask) & (~lac_absolute_mask)
print(f"Raw NPR absolute rows: {raw_npr_mask.sum()}")
print(df[raw_npr_mask]['PRICE'].head(5))

Raw NPR absolute rows: 208
4         Rs. 12000000
5         Rs. 27000000
140    Rs. 3,88,000,00
184     Rs. 27500000  
397     Rs. 27500000  
Name: PRICE, dtype: object


In [94]:
# NOw looking at rate rows only 

df_rate= df[rate_mask].copy()

# classifying rate rows by denominator
rate_aana_mask = df_rate['PRICE_CLEAN'].str.contains('/aana', na=False, case=False, regex=False)
rate_m_mask = df_rate['PRICE_CLEAN'].str.contains('/m', na=False, case=False, regex=False)

print(f"Rate /aana rows: {rate_aana_mask.sum()}")
print(f"Rate /m rows: {rate_m_mask.sum()}")
print(f"Rate other: {(~rate_aana_mask & ~rate_m_mask).sum()}")

print("\nRate /aana examples:")
print(df_rate[rate_aana_mask]['PRICE'].head(5))

print("\nRate /m examples:")
print(df_rate[rate_m_mask]['PRICE'].head(5))

Rate /aana rows: 26
Rate /m rows: 534
Rate other: 1

Rate /aana examples:
84      Rs. 50 Lac/aana
136     Rs. 90 Lac/aana
163     Rs. 70 Lac/aana
166     Rs. 21 Lac/aana
1019    Rs. 53 Lac/aana
Name: PRICE, dtype: object

Rate /m examples:
98      Rs. 65,000 /m
109     Rs. 1.4 Lac/m
111     Rs. 1.5 Lac/m
127    Rs. 1.05 Lac/m
159     Rs. 2.5 Lac/m
Name: PRICE, dtype: object


In [95]:
# finally verifying no overlap (all rows accounted)

total_accounted = (
    non_price_mask.sum() + 
    rate_mask.sum() + 
    cr_absolute_mask.sum() + 
    lac_absolute_mask.sum() + 
    raw_npr_mask.sum()
)

print(f"Total rows in original: {len(df)}")
print(f"Total rows classified: {total_accounted}")
print(f"Difference: {len(df) - total_accounted}")

if len(df) - total_accounted != 0:
    print("\nWARNING: Some rows not classified!")
    all_masks = non_price_mask | rate_mask | cr_absolute_mask | lac_absolute_mask | raw_npr_mask
    print(df[~all_masks]['PRICE'].head(10))

Total rows in original: 3418
Total rows classified: 3418
Difference: 0


# Finally all the classification have been done so i am moving to conversion

In [96]:
# Creating empty column for final NPR Values
df['PRICE_NPR'] = None

# Step1: Parse Cr absolute prices

In [101]:
# Identified which rows are Cr absolute (from earlier classification, which doesnot contain '/' but only 'Cr')
# Extracting just the number from PRICE_CLEAN THAT CONTAINS CR ABSOLUTE
# Example -> Rs. 2.9Cr -> 2.9 and then multiplying by 10000000
#.loc select rows and columns by labels
cr_numbers = df.loc[cr_absolute_mask, 'PRICE_CLEAN'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
df.loc[cr_absolute_mask, 'PRICE_NPR'] = cr_numbers * 10000000
print(df[cr_absolute_mask][['PRICE', 'PRICE_NPR']].head(5))

          PRICE   PRICE_NPR
0   Rs. 2.9 Cr   29000000.0
1  Rs. 4.75 Cr   47500000.0
2  Rs. 1.99 Cr   19900000.0
3     Rs. 4 Cr   40000000.0
6   Rs. 3.3 Cr   33000000.0


# Step2: Parse Lac absolute prices

In [102]:
# alredy identified lac absolute rows
# extracting just jumbers from PRICE_CLEAN
# Rs. 1.5 Lac -> 1.5 and then multiplying by 100,000 to make it absolute number

lac_numbers = df.loc[lac_absolute_mask, 'PRICE_CLEAN'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
df.loc[lac_absolute_mask, 'PRICE_NPR'] = lac_numbers * 100000
print(df[lac_absolute_mask][['PRICE', 'PRICE_NPR']].head(10))


             PRICE  PRICE_NPR
210   Rs. 1.5 Lac    150000.0
357    Rs. 90 Lac   9000000.0
386    Rs. 65 Lac   6500000.0
546    Rs. 15 Lac   1500000.0
642   Rs. 1.1 Lac    110000.0
683    Rs. 60 Lac   6000000.0
870     Rs. 3 Lac    300000.0
1097    Rs. 1 Lac    100000.0
1308   Rs. 85 Lac   8500000.0
1656   Rs. 85 Lac   8500000.0


# Step3: Parse Raw Npr absolute prices 

In [ ]:
# actually we are converting  "Rs. 12000000" to float
# these are already in npr we just need to remove commas and convert into numbers
# already identified raw npr numbers 
# removing commas, then converting to float
# Example: "Rs. 3,65,00,000" -> remove commas -> "36500000" -> 36500000.0
raw_numbers = df.loc[raw_npr_mask, 'PRICE_CLEAN'].str.replace(',', '', regex=False).str.extract(r'(\d+)')[0].astype(float)
df.loc[raw_npr_mask, 'PRICE_NPR'] = raw_numbers

# Step5: Parse rate prices ( this is also connected with land area)

In [104]:
rate_rows = df[rate_mask].copy()
print(rate_rows['PRICE'].head(10))
print(rate_rows['PRICE'].unique()[:20])




84     Rs. 50 Lac/aana
98       Rs. 65,000 /m
109      Rs. 1.4 Lac/m
111      Rs. 1.5 Lac/m
127     Rs. 1.05 Lac/m
136    Rs. 90 Lac/aana
159      Rs. 2.5 Lac/m
163    Rs. 70 Lac/aana
166    Rs. 21 Lac/aana
168      Rs. 60,000 /m
Name: PRICE, dtype: object
['Rs. 50 Lac/aana' 'Rs. 65,000 /m' 'Rs. 1.4 Lac/m' 'Rs. 1.5 Lac/m'
 'Rs. 1.05 Lac/m' 'Rs. 90 Lac/aana' 'Rs. 2.5 Lac/m' 'Rs. 70 Lac/aana'
 'Rs. 21 Lac/aana' 'Rs. 60,000 /m' 'Rs. 3.5 Lac/m' 'Rs. 80,000 /m'
 'Rs. 70,000 /m' 'Rs. 45,000 /m' 'Rs. 1.6 Lac/m' 'Rs. 1.28 Lac/m'
 'Rs. 90,000 /m' 'Rs. 1.7 Lac/m' 'Rs. 1.85 Lac/m' 'Rs. 3 Lac/m']


In [106]:
# create columns for rate informations
df['RATE_VALUE'] = None # numeric rate value  ( 1.4 , 1.5 for 1.4 lac/m)
df['RATE_UNIT'] = None # unit for rate (lac or npr or cr)
df['RATE_DENOMINATOR'] = None # what it's per (m or aana)

# find rows that has /m (per meter)
rate_m = df[rate_mask & df['PRICE_CLEAN'].str.contains('/m', na=False)]
rate_aana = df[rate_mask & df['PRICE_CLEAN'].str.contains('/aana', na=False)]
rate_other = df[rate_mask & ~df['PRICE_CLEAN'].str.contains('/m|/aana', na=False)]
print(f"Rate /m rows: {len(rate_m)}")
print(f"Rate /aana rows: {len(rate_aana)}")
print(f"Rate other rows: {len(rate_other)}")

# Show examples of each
print("\nExamples of /m:")
print(rate_m['PRICE'].head(3))

print("\nExamples of /aana:")
print(rate_aana['PRICE'].head(3))

if len(rate_other) > 0:
    print("\nExamples of other:")
    print(rate_other['PRICE'].head(3))


Rate /m rows: 534
Rate /aana rows: 26
Rate other rows: 1

Examples of /m:
98     Rs. 65,000 /m
109    Rs. 1.4 Lac/m
111    Rs. 1.5 Lac/m
Name: PRICE, dtype: object

Examples of /aana:
84     Rs. 50 Lac/aana
136    Rs. 90 Lac/aana
163    Rs. 70 Lac/aana
Name: PRICE, dtype: object

Examples of other:
1583    Rs. 50 /sf
Name: PRICE, dtype: object


In [125]:
# Parse /m rows - (that has 'Lac')
#extract numbers, convert lac/m into npr

# first finding which /m rows have 'Lac'
m_with_lac = rate_m[rate_m['PRICE_CLEAN'].str.contains('Lac', na=False)]

#extract number and convert to NPR
extracted_numbers = df.loc[m_with_lac.index, 'RATE_VALUE'] = m_with_lac['PRICE_CLEAN'].str.extract(r'(\d+\.?\d*)')[0].astype(float)
converted_values = extracted_numbers * 100000

# set unit and denominator
df.loc[m_with_lac.index, 'RATE_VALUE'] = converted_values
df.loc[m_with_lac.index, 'RATE_UNIT'] = 'NPR'
df.loc[m_with_lac.index, 'RATE_DENOMINATOR'] = 'm'


In [126]:
# Parse /m rows - (that has no 'Lac')
# these are already npr, just need to remove comma just like before

m_without_lac = rate_m[~rate_m['PRICE_CLEAN'].str.contains('Lac', na=False, case=False)]
df.loc[m_without_lac.index, 'RATE_VALUE'] = m_without_lac['PRICE_CLEAN'].str.replace(',', '', regex=False).str.extract(r'(\d+)')[0].astype(float)
df.loc[m_without_lac.index, 'RATE_UNIT'] = 'NPR'
df.loc[m_without_lac.index, 'RATE_DENOMINATOR'] = 'm'


In [127]:
# Parse /anna rows - (all has 'Lac' so it might be easier then others)
#extrace number and convert to NPR
df.loc[rate_aana.index, 'RATE_VALUE'] = rate_aana['PRICE_CLEAN'].str.extract(r'(\d+\.?\d*)')[0].astype(float) * 100000
df.loc[rate_aana.index, 'RATE_UNIT'] = 'NPR'
df.loc[rate_aana.index, 'RATE_DENOMINATOR'] = 'aana'


In [128]:
#  Check all rate rows are parsed
total_rate_parsed = df['RATE_VALUE'].notna().sum()
expected_rate_rows = len(rate_m) + len(rate_aana)

print(f"Total rate rows parsed: {total_rate_parsed}")
print(f"Expected rate rows: {expected_rate_rows}")
print(f"Match: {total_rate_parsed == expected_rate_rows}")

# Show random sample of parsed rate rows
print("\nRandom sample of parsed rate rows:")
sample_rate = df[df['RATE_VALUE'].notna()].sample(50)
print(sample_rate[['PRICE', 'RATE_VALUE', 'RATE_UNIT', 'RATE_DENOMINATOR']])

Total rate rows parsed: 560
Expected rate rows: 560
Match: True

Random sample of parsed rate rows:
                PRICE RATE_VALUE RATE_UNIT RATE_DENOMINATOR
998     Rs. 1.5 Lac/m   150000.0       NPR                m
1663   Rs. 5.75 Lac/m   575000.0       NPR                m
293     Rs. 82,000 /m    82000.0       NPR                m
1567      Rs. 6 Lac/m   600000.0       NPR                m
1178    Rs. 55,000 /m    55000.0       NPR                m
379     Rs. 2.5 Lac/m   250000.0       NPR                m
1446   Rs. 2.75 Lac/m   275000.0       NPR                m
1785    Rs. 75,000 /m    75000.0       NPR                m
2052      Rs. 3 Lac/m   300000.0       NPR                m
1019  Rs. 53 Lac/aana  5300000.0       NPR             aana
1347      Rs. 5 Lac/m   500000.0       NPR                m
558     Rs. 90,000 /m    90000.0       NPR                m
2490      Rs. 4 Lac/m   400000.0       NPR                m
551     Rs. 2.5 Lac/m   250000.0       NPR                m


In [133]:
# Check what's ACTUALLY in PRICE_NPR column
print("Unique values in PRICE_NPR (first 20):")
print(df['PRICE_NPR'].head(20))

print("\nData types of first 5 values:")
for i in range(5):
    val = df['PRICE_NPR'].iloc[i]
    print(f"Row {i}: {val} -> type: {type(val)}")

Unique values in PRICE_NPR (first 20):
0     29000000.0
1     47500000.0
2     19900000.0
3     40000000.0
4           None
5           None
6     33000000.0
7     40000000.0
8     45000000.0
9     30000000.0
10    49900000.0
11    24500000.0
12    25000000.0
13    42500000.0
14    65000000.0
15    36000000.0
16    58000000.0
17    25000000.0
18    48000000.0
19    66500000.0
Name: PRICE_NPR, dtype: object

Data types of first 5 values:
Row 0: 29000000.0 -> type: <class 'float'>
Row 1: 47500000.0 -> type: <class 'float'>
Row 2: 19900000.0 -> type: <class 'float'>
Row 3: 40000000.0 -> type: <class 'float'>
Row 4: None -> type: <class 'NoneType'>


In [134]:
# Convert PRICE_NPR to numeric (coerce errors to NaN)
df['PRICE_NPR'] = pd.to_numeric(df['PRICE_NPR'], errors='coerce')

# Convert RATE_VALUE to numeric
df['RATE_VALUE'] = pd.to_numeric(df['RATE_VALUE'], errors='coerce')

# Check dtypes again
print(df[['PRICE_NPR', 'RATE_VALUE']].dtypes)

PRICE_NPR     float64
RATE_VALUE    float64
dtype: object


In [135]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3418 entries, 0 to 3417
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   TITLE             3418 non-null   object 
 1   LOCATION          3418 non-null   object 
 2   PRICE             3418 non-null   object 
 3   LAND AREA         3329 non-null   object 
 4   BUILDUP AREA      719 non-null    object 
 5   ROAD ACCESS       3409 non-null   object 
 6   FACING            3212 non-null   object 
 7   FLOOR             3323 non-null   float64
 8   BEDROOM           3136 non-null   float64
 9   BATHROOM          3072 non-null   float64
 10  BUILT YEAR        3357 non-null   object 
 11  PARKING           632 non-null    object 
 12  AMENITIES         3418 non-null   object 
 13  PRICE_CLEAN       3418 non-null   object 
 14  PRICE_NPR         2459 non-null   float64
 15  RATE_VALUE        560 non-null    float64
 16  RATE_UNIT         560 non-null    object 
